In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import torchvision.datasets as datasets
from torchvision import models, transforms
from torchvision.utils import save_image, make_grid
from torch.optim.lr_scheduler import StepLR
from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter

from typing import Dict, Tuple
from tqdm import tqdm
import numpy as np
import time
import os
import random
from tabulate import tabulate

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

%matplotlib inline

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")



Torch version: 2.7.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4090


In [17]:
from waveguide_dataset_paired import WaveguideDataset
dataset = WaveguideDataset('train_test_split.h5')

In [18]:
class Flatten(nn.Module):
    def forward(self, x):
        return torch.flatten(x, 1)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CondEncoderWeighted(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=32, out_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU()
        )
        self.attn = nn.Sequential(
            nn.Linear(out_dim, 1)  # attention score for each pair
        )

    def forward(self, x):
        """
        x: [B, 4, 2] - 4 (mode, weight) pairs per sample
        returns: [B, out_dim] - pooled representation of pairs
        """
        h = self.encoder(x)                      # [B, 4, out_dim]
        attn_scores = self.attn(h).squeeze(-1)   # [B, 4]
        attn_weights = F.softmax(attn_scores, dim=1)  # [B, 4]
        attn_weights = attn_weights.unsqueeze(-1)     # [B, 4, 1]
        pooled = (h * attn_weights).sum(dim=1)        # [B, out_dim]
        return pooled

In [ ]:
class Net4_Mode0Weight0(nn.Module):
    """
    Predicts only the 0th mode-weight pair [mode0, weight0] using
    structured conditional input [B, 4, 2] and a single output head [B, 2].
    Uses GroupNorm for conv layers and LayerNorm for FC layers.
    """
    def __init__(self):
        super().__init__()

        # ---------- CNN trunk with GroupNorm ----------
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv2d(64, 128, 3, 1, 1), nn.GroupNorm(8, 128), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(128, 256, 3, 1, 1), nn.GroupNorm(16, 256), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(256, 512, 3, 1, 1), nn.GroupNorm(32, 512), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            Flatten()  # [B, 8192]
        )

        # ---------- Conditional encoder ----------
        self.cond_encoder = CondEncoderWeighted(in_dim=2, hidden_dim=32, out_dim=128)

        # ---------- Fully-connected trunk with LayerNorm ----------
        self.fc = nn.Sequential(
            nn.Linear(8192 + 128, 2048), nn.LayerNorm(2048), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(2048, 1024),       nn.LayerNorm(1024), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(1024, 256),        nn.LayerNorm(256),  nn.GELU(), nn.Dropout(0.25),
            nn.Linear(256, 2)  # Joint prediction: [mode0, weight0]
        )

    def forward(self, x_img, x_cond):
        """
        x_img: [B, 1, 32, 32]
        x_cond: [B, 4, 2] - 4 mode-weight pairs
        """
        img_feat = self.cnn(x_img)                 # [B, 8192]
        cond_feat = self.cond_encoder(x_cond)      # [B, 128]
        x = torch.cat((img_feat, cond_feat), dim=1)  # [B, 8320]
        return self.fc(x)                          # [B, 2]


In [ ]:
def train(model, device, loader, optimizer, loss_fn):
    """
    Train for one epoch.

    · model input: waveguide [B, 1, 32, 32], cond [B, 4, 2]
    · model output: [mode0, weight0] → shape [B, 2]
    · ground truth: 0th pair from cond ⇒ cond[:, 0, :]  → shape [B, 2]
    """
    model.train()

    for cond, params, waveguide in loader:
        # move to device
        cond = cond.to(device)              # [B, 4, 2]
        waveguide = waveguide.to(device)    # [B, 1, 32, 32]

        # ground truth: the 0th mode-weight pair
        y_true = cond[:, 0, :]              # [B, 2]

        # forward / backward
        optimizer.zero_grad()
        y_pred = model(waveguide, cond)     # [B, 2]
        loss = loss_fn(y_pred, y_true)
        loss.backward()
        optimizer.step()


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import torch

def test(model, device, loader, loss_fn, dataset, epoch_num, total_epochs):
    """
    Evaluate model for one epoch.

    * Model outputs   → shape [B, 2]   (mode-0, weight-0)
    * target is       → shape [B, 4, 2]; we extract target[:, 0, :] for comparison.

    Returns
    -------
    float
        Average MSE loss over the test set.
    """
    model.eval()
    total_loss = 0.0
    collected  = []  # store up to 50 (target, output, params) triplets

    # fetch scalar stats for denormalization
    mode_mean_log = float(dataset.meanm[0])
    mode_std_log  = float(dataset.stdm[0])
    wt_mean_log   = float(dataset.meanw[0])
    wt_std_log    = float(dataset.stdw[0])

    with torch.no_grad():
        for cond, params, waveguide in loader:
            cond = cond.to(device)           # [B, 4, 2]
            params = params.to(device)       # [B, 4]
            waveguide = waveguide.to(device) # [B, 1, 32, 32]

            # --- extract mode0 + weight0 ground truth ---
            y_true = cond[:, 0, :]  # [B, 2]

            # forward pass
            y_pred = model(waveguide, cond)  # [B, 2]

            # accumulate loss
            batch_loss = loss_fn(y_pred, y_true).item()
            total_loss += batch_loss * waveguide.size(0)

            # collect for plotting
            for t, o, p in zip(y_true.cpu(), y_pred.cpu(), params.cpu()):
                if len(collected) < 50:
                    # --- apply correct log-normalization inverse ---
                    t_mode0_log   = t[0].item() * mode_std_log + mode_mean_log
                    t_weight0_log = t[1].item() * wt_std_log + wt_mean_log
                    o_mode0_log   = o[0].item() * mode_std_log + mode_mean_log
                    o_weight0_log = o[1].item() * wt_std_log + wt_mean_log

                    t_mode0   = np.expm1(t_mode0_log)
                    t_weight0 = np.expm1(t_weight0_log)
                    o_mode0   = np.expm1(o_mode0_log)
                    o_weight0 = np.expm1(o_weight0_log)

                    collected.append((
                        np.array([t_mode0, t_weight0]),
                        np.array([o_mode0, o_weight0]),
                        p.numpy()
                    ))

    avg_loss = total_loss / len(loader.dataset)

    # ---------- plot on last epoch ----------
    if epoch_num == total_epochs - 1 and collected:
        chosen = random.sample(collected, 8)

        fig, (ax_mode, ax_weight) = plt.subplots(
            1, 2, figsize=(12, 4), sharex=True
        )

        # separate plots for modes and weights
        for tgt, out, prm in chosen:
            ax_mode.plot([0], [tgt[0]],  'ro')
            ax_mode.plot([0], [out[0]],  'bx')
            ax_weight.plot([0], [tgt[1]], 'ro')
            ax_weight.plot([0], [out[1]], 'bx')

        ax_mode.set_title("Mode-0")
        ax_weight.set_title("Weight-0")
        for ax in (ax_mode, ax_weight):
            ax.set_xticks([0])
            ax.set_xticklabels(['value'])
            ax.grid(True)

        plt.suptitle("Targets (red) vs Outputs (blue) on last epoch")
        plt.tight_layout()
        plt.show()

    return avg_loss


In [ ]:
def main(dataset):
    os.makedirs("models", exist_ok=True)
    
    batch_size = 128
    test_batch_size = 1000
    lr = 1e-3
    gamma = 0.9
    epochs = 200
    save_dir = 'only_top_mode_4x2'
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    model = Net4_Mode0Weight0().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = nn.MSELoss()

    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, num_workers=4)

    e_loss_graph = []

    # One progress bar over all epochs
    pbar = tqdm(range(epochs), desc="Training, loss= ----")

    for epoch in pbar:
        train(model, device, train_loader, optimizer, loss_fn)
        e_loss = test(model, device, test_loader, loss_fn, dataset, epoch, epochs)
        e_loss_graph.append(e_loss)
        pbar.set_description()
        scheduler.step()
        torch.save(model.state_dict(), f'models/{save_dir}.pth')
        pbar.set_description(f"Training, loss: {e_loss:.4f}")
        # Save loss graph after each epoch
        plt.figure()
        plt.plot(range(epoch + 1), e_loss_graph)
        plt.title(f"Epoch loss for {save_dir}")
        plt.xlabel("Epoch")
        plt.ylabel("Test Loss")
        plt.grid(True)
        plt.savefig(f"models/{save_dir}.png")
        plt.close()
if __name__ == '__main__':
    main(dataset)

Training, loss: 0.1118:  30%|███       | 61/200 [2:01:55<4:42:25, 121.91s/it]